In [ ]:
    #back to a previous version.

    #best code so far. 

    # Conditional Sequential VAE for 60-month yield-curve scenario generation
    # input file: raw_data.csv, now "raw_data_cleaned.csv"  since andreas asked to remove the gdp, cpi columns from the file.     this would have been better: df = pd.read_csv(FILE_PATH)   df = df.drop(columns=["FedFunds", "CPI", "GDP"], errors='ignore') # Minimal logic addition
    #

    # Expected columns:
    # DATE, Y_DGS1MO, Y_DGS3MO, Y_DGS6MO, Y_DGS1, Y_DGS2, Y_DGS3,
    # Y_DGS5, Y_DGS7, Y_DGS10, Y_DGS20, Y_DGS30


    #updated: normalisation and a validation split. we tried also rolling window split. 

    import os
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    import matplotlib.pyplot as plt

    # =========================
    # CONFIG
    # =========================

    FILE_PATH = "raw_data_cleaned.csv"
    OUTPUT_FILE = "18-5 4.0 (900 epochs) generated_yield_curve_scenarios.csv"

    YIELD_COLS = [
        "Y_DGS1MO", "Y_DGS3MO", "Y_DGS6MO",
        "Y_DGS1", "Y_DGS2", "Y_DGS3", "Y_DGS5",
        "Y_DGS7", "Y_DGS10", "Y_DGS20", "Y_DGS30"
    ]

    HORIZON = 60
    N_SCENARIOS = 200
    LATENT_DIM = 16
    HIDDEN_DIM = 256
    BATCH_SIZE = 32
    EPOCHS = 900      
    LEARNING_RATE = 1e-3

    BETA_KL = 0.01
    SMOOTH_LAMBDA = 0.10

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    torch.manual_seed(42)
    np.random.seed(42)

    # =========================
    # LOAD DATA
    # =========================
    df = pd.read_csv(FILE_PATH)

    df["DATE"] = pd.to_datetime(df["DATE"])
    df = df[["DATE"] + YIELD_COLS].copy()

    for col in YIELD_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().sort_values("DATE")

    monthly = (
        df.set_index("DATE")[YIELD_COLS]
        .resample("ME")
        .last()
        .dropna()
    )

    if len(monthly) <= HORIZON:
        raise ValueError(
            f"Not enough monthly data. Need more than {HORIZON} months, got {len(monthly)}."
        )

    curves = monthly.values.astype("float32")
    dates = monthly.index

    # =========================
    # CREATE TRAINING SAMPLES
    # =========================

    X = []
    y = []

    for i in range(len(curves) - HORIZON):
        current_curve = curves[i]

        future_curves = curves[i + 1 : i + HORIZON + 1]
        previous_curves = curves[i : i + HORIZON]

        monthly_changes = future_curves - previous_curves

        X.append(current_curve)
        y.append(monthly_changes)

    X = np.array(X, dtype="float32")
    y = np.array(y, dtype="float32")

    print("Current curve input shape:", X.shape)
    print("Future change target shape:", y.shape)


    monthly_train = monthly.loc[:'2020']
    monthly_val = monthly.loc['2021':]

    train_curves = monthly_train.values.astype("float32")
    val_curves = monthly_val.values.astype("float32")
    def create_samples(curves, horizon):
        X, y = [], []

        for i in range(len(curves) - horizon):
            current_curve = curves[i]

            future_curves = curves[i + 1 : i + horizon + 1]
            previous_curves = curves[i : i + horizon]

            monthly_changes = future_curves - previous_curves

            X.append(current_curve)
            y.append(monthly_changes)

        return (
            np.array(X, dtype="float32"),
            np.array(y, dtype="float32")
        )

    X_train, y_train = create_samples(train_curves, HORIZON)
    X_val, y_val = create_samples(val_curves, HORIZON)

    print("Train shape:", X_train.shape, y_train.shape)
    print("Validation shape:", X_val.shape, y_val.shape)
    # =========================
    # NORMALIZATION
    # =========================
    # here we have to do normalization better. prevent. we should have a validation split to better train the model and prevent overfitting. 
    # Calculate mean/std using ONLY training data to prevent data leakage
    x_mean, x_std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-6
    y_mean, y_std = y_train.reshape(-1, len(YIELD_COLS)).mean(axis=0), y_train.reshape(-1, len(YIELD_COLS)).std(axis=0) + 1e-6

    # Scale both sets using training parameters
    X_train_scaled, X_val_scaled = (X_train - x_mean) / x_std, (X_val - x_mean) / x_std
    y_train_scaled, y_val_scaled = (y_train - y_mean) / y_std, (y_val - y_mean) / y_std

    # Convert to tensors
    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_scaled), torch.tensor(y_train_scaled)), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_scaled), torch.tensor(y_val_scaled)), batch_size=BATCH_SIZE, shuffle=False)

    # =========================
    # MODEL
    # =========================

    class ConditionalYieldCurveVAE(nn.Module):
        def __init__(self, n_tenors, horizon=60, latent_dim=16, hidden_dim=256):
            super().__init__()

            self.n_tenors = n_tenors
            self.horizon = horizon
            self.output_dim = horizon * n_tenors

            encoder_input_dim = n_tenors + self.output_dim

            self.encoder = nn.Sequential(
                nn.Linear(encoder_input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            )

            self.mu = nn.Linear(hidden_dim, latent_dim)
            self.logvar = nn.Linear(hidden_dim, latent_dim)

            decoder_input_dim = n_tenors + latent_dim

            self.decoder = nn.Sequential(
                nn.Linear(decoder_input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, self.output_dim),
            )

        def encode(self, y0, future_changes):
            batch_size = y0.shape[0]
            flat_future = future_changes.reshape(batch_size, -1)
            x = torch.cat([y0, flat_future], dim=1)
            h = self.encoder(x)
            return self.mu(h), self.logvar(h)

        def reparameterize(self, mu, logvar):
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std

        def decode(self, y0, z):
            x = torch.cat([y0, z], dim=1)
            out = self.decoder(x)
            return out.reshape(-1, self.horizon, self.n_tenors)

        def forward(self, y0, future_changes):
            mu, logvar = self.encode(y0, future_changes)
            z = self.reparameterize(mu, logvar)
            pred_changes = self.decode(y0, z)
            return pred_changes, mu, logvar

    # =========================
    # LOSS
    # =========================

    def vae_loss(pred, target, mu, logvar):                                               # we should try a better loss function. 
        reconstruction_loss = F.mse_loss(pred, target)

        kl_loss = -0.5 * torch.mean(
            1 + logvar - mu.pow(2) - logvar.exp()
        )

        # Smoothness across maturity tenors
        tenor_diff = pred[:, :, 1:] - pred[:, :, :-1]
        smoothness_loss = torch.mean(tenor_diff ** 2)

        total_loss = (
            reconstruction_loss
            + BETA_KL * kl_loss
            + SMOOTH_LAMBDA * smoothness_loss
        )

        return total_loss, reconstruction_loss, kl_loss, smoothness_loss

    # =========================
    # TRAIN
    # =========================

    model = ConditionalYieldCurveVAE(
        n_tenors=len(YIELD_COLS),
        horizon=HORIZON,
        latent_dim=LATENT_DIM,
        hidden_dim=HIDDEN_DIM
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(1, EPOCHS + 1):
        model.train() # Set model to training mode
        for y0_batch, changes_batch in train_loader: # Iterate over training data
            optimizer.zero_grad() # Clear previous gradients
            pred, mu, logvar = model(y0_batch.to(DEVICE), changes_batch.to(DEVICE)) # Forward pass
            loss, _, _, _ = vae_loss(pred, changes_batch.to(DEVICE), mu, logvar) # Calculate loss
            loss.backward() # Backpropagate
            optimizer.step() # Update weights

        # Validation phase
        model.eval() # Set model to evaluation mode
        val_loss = 0
        with torch.no_grad(): # Disable gradient calculation
            for y0_v, changes_v in val_loader: # Iterate over validation data
                pred, mu, logvar = model(y0_v.to(DEVICE), changes_v.to(DEVICE))
                loss, _, _, _ = vae_loss(pred, changes_v.to(DEVICE), mu, logvar)
                val_loss += loss.item()
                
        if epoch % 50 == 0 or epoch == 1: # Print status
            print(f"Epoch {epoch:04d} | Train Loss: {loss.item():.6f} | Val Loss: {val_loss/len(val_loader):.6f}")
    # =========================
    # GENERATE SCENARIOS
    # =========================
    def generate_scenarios(model, current_curve, n_scenarios=200):
        model.eval()

        current_curve = np.array(current_curve, dtype="float32")
        current_scaled = (current_curve - x_mean) / x_std

        y0 = torch.tensor(current_scaled, dtype=torch.float32)
        y0 = y0.unsqueeze(0).repeat(n_scenarios, 1).to(DEVICE)

        z = torch.randn(n_scenarios, LATENT_DIM).to(DEVICE)

        with torch.no_grad():
            pred_scaled_changes = model.decode(y0, z).cpu().numpy()

        pred_changes = pred_scaled_changes * y_std + y_mean

        projected_curves = (
            current_curve.reshape(1, 1, len(YIELD_COLS))
            + np.cumsum(pred_changes, axis=1)
        )

        return projected_curves

    current_curve = monthly.iloc[-1].values.astype("float32")
    last_date = monthly.index[-1]

    scenarios = generate_scenarios(
        model=model,
        current_curve=current_curve,
        n_scenarios=N_SCENARIOS
    )

    print("Generated scenario shape:", scenarios.shape)

    # =========================
    # EXPORT CSV
    # =========================

    projection_dates = pd.date_range(
        last_date + pd.offsets.MonthEnd(1),
        periods=HORIZON,
        freq="ME"
    )

    rows = []

    for scenario_id in range(N_SCENARIOS):
        for month_idx in range(HORIZON):
            row = {
                "scenario_id": scenario_id + 1,
                "projection_month": month_idx + 1,
                "projection_date": projection_dates[month_idx].date()
            }

            for tenor_idx, col in enumerate(YIELD_COLS):
                row[col] = scenarios[scenario_id, month_idx, tenor_idx]

            rows.append(row)

    scenario_df = pd.DataFrame(rows)
    scenario_df.to_csv(OUTPUT_FILE, index=False)

    print(f"Saved generated scenarios to: {OUTPUT_FILE}")

    # =========================
    # SIMPLE VALIDATION SUMMARY
    # =========================

    historical_changes = y.reshape(-1, len(YIELD_COLS))
    generated_changes = np.diff(
        np.concatenate(
            [
                current_curve.reshape(1, 1, len(YIELD_COLS)).repeat(N_SCENARIOS, axis=0),
                scenarios
            ],
            axis=1
        ),
        axis=1
    ).reshape(-1, len(YIELD_COLS))




    #here we want a plot of the training and validation loss, so we can prevent overfitting.
    #there should be a way to stop and save the best model to use for scenario generation.


    #performance on 500 epochs was a lot higher than 50 eventhough val loss started to go up from there.  700 epochs was best so far.  perhaps try learning scheduler??

Current curve input shape: (230, 11)
Future change target shape: (230, 60, 11)
Train shape: (168, 11) (168, 60, 11)
Validation shape: (2, 11) (2, 60, 11)
Epoch 0001 | Train Loss: 1.546698 | Val Loss: 1.546698
Epoch 0050 | Train Loss: 1.581390 | Val Loss: 1.581390
Epoch 0100 | Train Loss: 1.591525 | Val Loss: 1.591525
Epoch 0150 | Train Loss: 1.587823 | Val Loss: 1.587823
Epoch 0200 | Train Loss: 1.572161 | Val Loss: 1.572161
Epoch 0250 | Train Loss: 1.570791 | Val Loss: 1.570791
Epoch 0300 | Train Loss: 1.570235 | Val Loss: 1.570235
Epoch 0350 | Train Loss: 1.581500 | Val Loss: 1.581500
Epoch 0400 | Train Loss: 1.578855 | Val Loss: 1.578855
Epoch 0450 | Train Loss: 1.581975 | Val Loss: 1.581975
Epoch 0500 | Train Loss: 1.598415 | Val Loss: 1.598415
Epoch 0550 | Train Loss: 1.549042 | Val Loss: 1.549042
Epoch 0600 | Train Loss: 1.594154 | Val Loss: 1.594154
Epoch 0650 | Train Loss: 1.553719 | Val Loss: 1.553719
Epoch 0700 | Train Loss: 1.561566 | Val Loss: 1.561566
Epoch 0750 | Train Lo

In [7]:
# ==========================================
# GENERATE FROM 10 DIFFERENT STARTING CURVES
# ==========================================

INITIAL_CURVES_FILE = "sampled_yield_curves (1).csv"
OUTPUT_FILE = "18-5 VAE_generated_scenarios_from_10_starting_curves_20each.csv"

SCENARIOS_PER_CURVE = 20

sampled_df = pd.read_csv(INITIAL_CURVES_FILE)

sampled_to_model_cols = {
    "Yield_1M": "Y_DGS1MO",
    "Yield_3M": "Y_DGS3MO",
    "Yield_6M": "Y_DGS6MO",
    "Yield_1Y": "Y_DGS1",
    "Yield_2Y": "Y_DGS2",
    "Yield_3Y": "Y_DGS3",
    "Yield_5Y": "Y_DGS5",
    "Yield_7Y": "Y_DGS7",
    "Yield_10Y": "Y_DGS10",
    "Yield_20Y": "Y_DGS20",
    "Yield_30Y": "Y_DGS30",
}

sampled_df = sampled_df.rename(columns=sampled_to_model_cols)

# Keep first 10 starting curves
sampled_df = sampled_df.head(10)

# Ensure same tenor order as VAE training
sampled_df = sampled_df[YIELD_COLS].copy()

all_rows = []

print(f"Generating scenarios from {len(sampled_df)} starting curves...")

global_scenario_id = 1

for start_curve_id, row in sampled_df.iterrows():

    print(f"Processing starting curve {start_curve_id + 1}/{len(sampled_df)}")

    current_curve = row.values.astype("float32")

    scenarios = generate_scenarios(
        model=model,
        current_curve=current_curve,
        n_scenarios=SCENARIOS_PER_CURVE
    )

    rows = []

    for scenario_id in range(SCENARIOS_PER_CURVE):
        for month_idx in range(HORIZON):

            output_row = {
                "Start_Curve_ID": start_curve_id + 1,
                "Scenario_ID": global_scenario_id + scenario_id,
                "scenario_id_within_start_curve": scenario_id + 1,
                "Month": month_idx + 1,
            }

            for tenor_idx, col in enumerate(YIELD_COLS):
                output_row[col] = scenarios[scenario_id, month_idx, tenor_idx]

            rows.append(output_row)

    temp_df = pd.DataFrame(rows)
    all_rows.append(temp_df)

    global_scenario_id += SCENARIOS_PER_CURVE

final_df = pd.concat(all_rows, ignore_index=True)

final_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved combined scenarios to: {OUTPUT_FILE}")
print(f"Number of starting curves: {len(sampled_df)}")
print(f"Scenarios per starting curve: {SCENARIOS_PER_CURVE}")
print(f"Total unique scenarios: {final_df['Scenario_ID'].nunique()}")
print(f"Total rows: {len(final_df)}")

Generating scenarios from 10 starting curves...
Processing starting curve 1/10
Processing starting curve 2/10
Processing starting curve 3/10
Processing starting curve 4/10
Processing starting curve 5/10
Processing starting curve 6/10
Processing starting curve 7/10
Processing starting curve 8/10
Processing starting curve 9/10
Processing starting curve 10/10

Saved combined scenarios to: 18-5 VAE_generated_scenarios_from_10_starting_curves_20each.csv
Number of starting curves: 10
Scenarios per starting curve: 20
Total unique scenarios: 200
Total rows: 12000
